# QuantStream — fine-tune Chronos-2 bằng LoRA trên Kaggle

Add Input chứa `QuantStream_Kaggle_Input.zip`, bật Internet và GPU khả dụng. Notebook tải checkpoint công khai `amazon/chronos-2`, pin revision thực tế, fine-tune LoRA bằng log-price quá khứ và so sánh trước/sau trên cùng validation windows.

**Giới hạn:** chưa xác minh dữ liệu pretraining có trùng giai đoạn tài chính đang đánh giá. Đây là thí nghiệm học fine-tuning, không phải bằng chứng alpha độc lập hoặc kết quả competition.

In [ ]:
import subprocess,sys,os
os.environ['HF_HUB_DISABLE_XET']='1'
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'chronos-forecasting==2.3.2','peft>=0.18.1,<1','accelerate>=1.1,<2',
    'datasets>=3,<5','pyarrow>=17,<27','scikit-learn>=1.5,<2'])

In [ ]:
from pathlib import Path
import sys, zipfile, shutil, json

WORK = Path('/kaggle/working/quantstream') if Path('/kaggle/working').exists() else Path.cwd()/'notebook_work'
WORK.mkdir(parents=True,exist_ok=True)
INPUT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd().parent
archives = list(INPUT.rglob('QuantStream_Kaggle_Input.zip'))
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            p=Path(name)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError('Invalid archive path')
        z.extractall(WORK)
else:
    # Kaggle may automatically extract a ZIP uploaded as a Dataset.
    manifests=[p for p in INPUT.rglob('manifest.json') if (p.parent/'development.parquet').exists()]
    if len(manifests)!=1:
        raise FileNotFoundError('Attach the Dataset containing QuantStream_Kaggle_Input.zip (or its extracted contents).')
    source=manifests[0].parent
    shutil.copytree(source,WORK/'snapshot',dirs_exist_ok=True)
    shutil.copytree(source.parent/'code',WORK/'code',dirs_exist_ok=True)
SNAPSHOT=WORK/'snapshot'
sys.path.insert(0,str(WORK/'code'))
manifest=json.loads((SNAPSHOT/'manifest.json').read_text())
assert manifest['contains_final_holdout'] is False
print('Snapshot:',SNAPSHOT)
print('Final holdout included:',manifest['contains_final_holdout'])

In [ ]:
import torch
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
# Nếu GPU chưa có, hãy chọn accelerator trong Settings trước khi train.
# Hàm hỗ trợ CPU cho smoke run; GPU thuận tiện hơn cho lần train đầy đủ.

## Fine-tuning thật

LoRA cập nhật các adapter trọng số; khác với thử learning rate hoặc threshold. PEFT phải cài thành công; mã không chấp nhận âm thầm chuyển sang full fine-tuning khi thiếu PEFT. Dữ liệu train dừng trước validation. Future covariates không chứa giá tương lai.

200 steps, context 128 và batch size 8 là cấu hình nhỏ để chạy thử. Không đổi cấu hình vì xem kết quả final holdout.

In [ ]:
from quantstream.finetune import chronos_experiment
OUTPUT=WORK/'chronos_output'
report=chronos_experiment(SNAPSHOT,OUTPUT,steps=200,max_windows=512,device='auto')
print(json.dumps(report,indent=2))

## Đọc kết quả và lưu adapter

MAE thấp hơn mới là cải thiện sai số dự báo; directional accuracy tăng không tự động có nghĩa chiến lược có lãi sau phí. Adapter không tự ghi đè champion local. Output kèm base revision, snapshot hash, số cửa sổ và các hạn chế để bạn viết báo cáo đúng.

In [ ]:
import pandas as pd
from IPython.display import display,FileLink
display(pd.DataFrame([
 {'mode':'zero_shot','mae':report['zero_shot_mae'],'directional_accuracy':report['zero_shot_directional_accuracy']},
 {'mode':'LoRA','mae':report['finetuned_mae'],'directional_accuracy':report['finetuned_directional_accuracy']}
]))
# Chỉ xuất adapter và evidence, không cần ZIP optimizer checkpoints.
EXPORT=WORK/'chronos_export'
EXPORT.mkdir(exist_ok=True)
shutil.copytree(OUTPUT/'adapter',EXPORT/'adapter',dirs_exist_ok=True)
for name in ['chronos_report.json','chronos_validation.parquet','training_snapshot_manifest.json']:
 shutil.copy2(OUTPUT/name,EXPORT/name)
archive=shutil.make_archive(str(WORK.parent/'Chronos_Kaggle_Output'),'zip',EXPORT)
display(FileLink(archive))